In [ ]:
from src.mlcore.dataloader import get_processed_data

raw = get_processed_data(
    source="local",
    symbol="BTCUSDC",
    save_dir="parquets/",
    grid_resolution_ms=5000,
)
raw

In [ ]:
from src.mlcore.dataloader import calculate_features

df = calculate_features(raw, filename='all_features')
df['open'] = raw['open']
df['high'] = raw['high']
df['low'] = raw['low']
df['close'] = raw['close']
df['volume'] = raw['volume']
df.dropna(inplace=True)
df

In [ ]:
feature_columns = [col for col in df.columns if col not in ['open', 'high', 'low', 'close', 'volume']]
X = df[feature_columns].copy()
ohlcv = df[['open', 'high', 'low', 'close', 'volume']].copy()

split_index = int(len(X) * 0.75)
_, X_test = X.iloc[:split_index], X.iloc[split_index:]
_, ohlcv_test = ohlcv.iloc[:split_index], ohlcv.iloc[split_index:]

In [ ]:
from xgboost import XGBRegressor
from src.trading.backtest_trader import BacktestTrader

mid_model = XGBRegressor()
mid_model._estimator_type = "regressor"
mid_model.load_model('models/model_mid.json')

spread_model = XGBRegressor()
spread_model._estimator_type = "regressor"
spread_model.load_model('models/model_spread.json')

mid_pred = mid_model.predict(X_test)
spread_pred = spread_model.predict(X_test)

In [ ]:
import matplotlib.pyplot as plt
from src.trading.backtest_trader import BacktestTrader, optimize_parameters_fast, optimize_parameters

bt = BacktestTrader(
    alpha=5.126745922262708,
    beta=7.853469588273192,
    gamma=1.0,
    epsilon=0.0,
    max_position=0.22,
    position_qty=0.004,
)

# bar_seconds должен соответствовать сетке данных: 5000 ms -> 5.0, 1000 ms -> 1.0
results = bt.run(
    ohlcv=ohlcv_test,
    mid_predicts=mid_pred,
    spread_predicts=spread_pred,
    position_size=1000.0,
    bar_seconds=5.0,
)
print(f"Sharpe: {results['sharpe_ratio']}")
plt.figure(figsize=(14,7))
plt.plot(results['bh_curve'], color="black")
plt.plot(results['realized_pnl_curve'], color="red")
plt.grid()
plt.show()
plt.figure(figsize=(14, 7))
plt.plot(results['position_curve'], color="blue")
plt.grid()
plt.show()

### Bayessian Optimization

In [ ]:
opt_results = optimize_parameters_fast(
    ohlcv=ohlcv_test,
    mid_predicts=mid_pred,
    spread_predicts=spread_pred,
    position_size=100.0,
    max_position=0.22,
    position_qty=0.004,
    alpha_range=(0, 7),
    beta_range=(-10, 10),
    gamma_range=(0, 1),
    epsilon_range=(0, 0.02),
    n_calls=450,
    n_initial_points=150,
)

import json

# Сохранение в файл
with open('opt_results.json', 'w', encoding='utf-8') as f:
    json.dump(opt_results, f, ensure_ascii=False, indent=4)

### Grid Search

In [ ]:
# _ = optimize_parameters(
#     ohlcv=ohlcv_test,
#     mid_predicts=mid_pred,
#     spread_predicts=spread_pred,
#     position_size=100.0,
#     alpha_range=(0, 5, 0.1),
#     beta_range=(0, 5, 0.1),
#     gamma_range=(0.1, 0.1, 0.1),
#     epsilon_range=(0.0, 0.0, 0.001),
#     max_position=0.02,
#     position_qty=0.002,
# )